In [ ]:
#| default_exp model.models

In [8]:
#| export 
import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
from omegaconf import DictConfig
from typing import List, Tuple
from einops import rearrange
from t2i_reid.model.pos_embeddings import sincos_2d
import math

def make_model(
    config: DictConfig,
):

    if config.model.name == "clip_vitb_16":
        
        model, preprocess = clip.load(
            "ViT-B/16",
            device=config.device,
            jit=False,
        )

        return model, preprocess

In [9]:
#| export
def encode_image_tokens(model, image):

    """All visual tokens of a CLIP ViT: [B, 1 + grid**2, embed_dim]. Token 0 is CLS (== model.encode_image)."""

    v = model.visual
    x = v.conv1(image.type(model.dtype))                                   # [B, width, grid, grid]
    x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)             # [B, grid**2, width]
    cls = v.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
    x = torch.cat([cls, x], dim=1) + v.positional_embedding.to(x.dtype)
    x = v.ln_pre(x)
    x = v.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)                 # NLD -> LND -> NLD
    x = v.ln_post(x)                                                       # all tokens, not just x[:, 0]
    return x @ v.proj


def encode_text_tokens(model, text):

    """All text tokens of CLIP: [B, context_length, embed_dim]. The global feature (== model.encode_text) is the
    EOT token at text.argmax(-1); positions after it are padding."""
    
    x = model.token_embedding(text).type(model.dtype) + model.positional_embedding.type(model.dtype)
    x = model.transformer(x.permute(1, 0, 2)).permute(1, 0, 2)             # NLD -> LND -> NLD
    x = model.ln_final(x).type(model.dtype)
    return x @ model.text_projection                                       # all positions, not just EOT

In [10]:
#| export
class CrossAttention(nn.Module):

    def __init__(
        self,
        embed_dim: int,
        num_heads: int = 8,
        query_dim: int = 512,
        context_dim: int = 512,
        dropout: float = 0.6,
    ):

        super().__init__()

        self.embed_dim = embed_dim
        self.h = num_heads
        self.dropout = nn.Dropout(dropout)
        
        assert self.embed_dim % self.h == 0, f"Number of heads must divide the embed_dim. got embed_dim - {embed_dim}, num_heads - {num_heads}"

        self.query_projection = nn.Linear(
            in_features = query_dim,
            out_features = embed_dim,
        ) # Q (B, query_dim) -> (B, hidden_dim)

        self.key_projection = nn.Linear(
            in_features = context_dim,
            out_features = embed_dim,
        ) # K (B, query_dim) -> (B, hidden_dim)

        self.value_projection = nn.Linear(
            in_features = context_dim,
            out_features = embed_dim,
        ) # V (B, query_dim) -> (B, hidden_dim)

        self.out_projection = nn.Linear(
            in_features = embed_dim,
            out_features = query_dim,
        ) # Output projection

        self.scale = (embed_dim // num_heads) ** -0.5  # 1/sqrt(per-head dim)

    def forward(
        self,
        query: torch.Tensor,
        context: torch.Tensor,
        context_mask: torch.Tensor = None,
        return_attn: bool = False,
    ):  

        q, k, v = self.query_projection(query), self.key_projection(context), \
             self.value_projection(context)

        q = rearrange(
            q,
            "b n (h d) -> b h n d",
            h = self.h
        )

        k = rearrange(
            k,
            "b n (h d) -> b h n d",
            h = self.h
        )

        v = rearrange(
            v,
            "b n (h d) -> b h n d",
            h = self.h
        )

        # Compute attention
        attn_logits = torch.matmul(
            q,
            k.transpose(-1, -2),
        ) * self.scale # (B, H, Nq, Nk)

        if context_mask is not None:

            mask = rearrange(
                context_mask,
                "b n -> b 1 1 n",
            )

            attn_logits = attn_logits.masked_fill(
                ~mask,
                torch.finfo(attn_logits.dtype).min,
            )

        attn = F.softmax(
            attn_logits,
            dim=-1
        )

        attn = self.dropout(attn)
        out = torch.matmul(attn, v)

        out = rearrange(
            out,
            "b h n d -> b n (h d)",
            h=self.h
        )

        out = self.out_projection(out)

        if return_attn:
            return out, attn

        return out

class CrossAttentionBlock(nn.Module):

    def __init__(
        self,
        query_dim: int ,
        context_dim: int,
        embed_dim: int,
        num_heads: int,
        mlp_ratio: float,
        dropout: float,
    ):

        super().__init__()

        self.norm1 = nn.LayerNorm(query_dim)

        self.cross_attn = CrossAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            query_dim=query_dim,
            context_dim=context_dim,
            dropout=dropout
        )

        self.norm2 = nn.LayerNorm(query_dim)

        hidden_dim = int(query_dim * mlp_ratio)

        self.mlp = nn.Sequential(
            nn.Linear(query_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, query_dim),
            nn.Dropout(dropout),
        )

    def forward(
        self,
        query: torch.Tensor,
        context: torch.Tensor,
        context_mask: torch.Tensor | None = None,
        return_attention: bool = False,
    ):

        residual = query

        query_norm = self.norm1(query)

        if return_attention:
            attn_out, attn = self.cross_attn(
                query=query_norm,
                context=context,
                context_mask=context_mask,
                return_attention=True,
            )
        else:
            attn_out = self.cross_attn(
                query=query_norm,
                context=context,
                context_mask=context_mask,
            )

        query = residual + attn_out

        query = query + self.mlp(
            self.norm2(query)
        )

        if return_attention:
            return query, attn

        return query

class CrossAttentionTransformer(nn.Module):

    def __init__(
        self,
        query_dim: int,
        context_dim: int,
        embed_dim: int,
        dropout: float = 0.0,
        depth: int = 4,
        mlp_ratio: float = 4.0,
        num_heads: int = 8,
    ):

        super().__init__()

        self.blocks = nn.ModuleList([
            CrossAttentionBlock(
                query_dim = query_dim,
                context_dim = context_dim,
                embed_dim = embed_dim,
                num_heads = num_heads,
                dropout = dropout,
                mlp_ratio = mlp_ratio
            )
        for i in range(depth)
        ])

    def forward(
        self,
        query: torch.Tensor,
        context: torch.Tensor,
        context_mask: torch.Tensor = None,
        return_attention: bool = False,
    ):

        attentions = []

        for block in self.blocks:

            if return_attention:
                query, attn = block(
                    query = query,
                    context = context,
                    context_mask = context_mask,
                    return_attention = return_attention,
                )

                attentions.append(attn)

            else:
                query = block(
                    query = query,
                    context = context,
                    context_mask = context_mask,
                    return_attention = return_attention
                )

        if return_attention:
            return query, attentions

        return query

In [11]:
query = torch.rand(1, 77, 512)
context = torch.rand(1, 197, 512)

cross_attn_transformer = CrossAttentionTransformer(
    query_dim=512,
    context_dim=512,
    embed_dim=512,
)

In [12]:
enriched_query = cross_attn_transformer.forward(query, context, None, False)

In [13]:
#| export
class SpatialTokens(nn.Module):

    def __init__(
        self,
        grid_size: int = 14,
        dim: int = 512,
    ):

        """C: one learned token per patch query, plus each query's reference point (patch centre, normalised (x, y))."""
        super().__init__()

        self.grid_size = grid_size
        self.tokens = nn.Parameter(sincos_2d(grid_size, dim) + 0.02 * torch.randn(grid_size**2, dim))
        ys, xs = torch.meshgrid(torch.arange(grid_size), torch.arange(grid_size), indexing="ij")
        ref = torch.stack([(xs + 0.5) / grid_size, (ys + 0.5) / grid_size], dim=-1).reshape(-1, 2)
        self.register_buffer('reference_points', ref, persistent=False)

    def forward(
        self,
        batch_size: int
    ):

        return self.tokens.unsqueeze(0).expand(batch_size, -1, -1)

In [15]:
#| export 
class OffsetHead(nn.Module):

    def __init__(
        self,
        dim: int = 512,
        num_heads: int = 8,
        num_points: int = 64,
    ):

        super().__init__()
        self.h, self.p = num_heads, num_points
        self.offsets = nn.Linear(dim, num_heads * num_points * 2)
        self.weights = nn.Linear(dim, num_heads * num_points)

        self._reset_parameters()

    def _reset_parameters(
        self,
    ):

        nn.init.zeros_(self.offsets.weight)
        theta = torch.arange(self.h) * (2 * math.pi / self.h)
        ring = torch.stack([theta.cos(), theta.sin()], dim=-1)
        ring = ring / ring.abs().max(dim=-1, keepdim=True).values                # onto the unit square
        ring = ring[:, None] * torch.arange(1, self.p + 1)[None, :, None]        # [H, P, 2]
        with torch.no_grad():
            self.offsets.bias.copy_(ring.flatten())                              # flatten order == 'h p xy'
        nn.init.zeros_(self.weights.weight)
        nn.init.zeros_(self.weights.bias)

    def forward(
        self,
        c: torch.Tensor
    ):

        offsets = rearrange(self.offsets(c), "b q (h p xy) -> b q h p xy", h=self.h, p=self.p)
        weights = rearrange(self.weights(c), "b q (h p) -> b q h p", h=self.h).softmax(dim=-1)
        return offsets, weights

In [16]:
#| export
class TextGuidedOffsets(nn.Module):

    """Text enriched by the image predicts, for each of the 196 patch queries, where its H×P samples should land."""

    def __init__(
        self, 
        dim=512, 
        grid_size=14, 
        num_heads=8, 
        num_points=4, 
        text_depth=4, 
        spatial_depth=2, 
        dropout=0.
    ):

        super().__init__()
        self.grid_size = grid_size
        self.text_enricher = CrossAttentionTransformer(query_dim=dim, context_dim=dim, embed_dim=dim,
                                                       dropout=dropout, depth=text_depth, num_heads=num_heads)
        self.spatial_tokens = SpatialTokens(grid_size, dim)
        self.spatial_decoder = CrossAttentionTransformer(query_dim=dim, context_dim=dim, embed_dim=dim,
                                                         dropout=dropout, depth=spatial_depth, num_heads=num_heads)
        self.norm = nn.LayerNorm(dim)                       # blocks are pre-LN, so normalise before the head
        self.offset_head = OffsetHead(dim, num_heads, num_points)

    def forward(
        self, 
        text_tokens, 
        text_mask, 
        image_tokens
    ):

        patches = image_tokens[:, 1:]                                               # [B, 196, d], drop CLS
        text = self.text_enricher(text_tokens, patches)                             # [B, 77, d]
        c = self.spatial_tokens(text.shape[0])                                      # [B, 196, d]
        c = self.spatial_decoder(c, text, context_mask=text_mask)                   # C̃: padding words masked
        offsets, weights = self.offset_head(self.norm(c))
        ref = self.spatial_tokens.reference_points[None, :, None, None]             # [1, 196, 1, 1, 2]
        locations = ref + offsets / self.grid_size                                  # [B, 196, H, P, 2] in [0, 1]
        return locations, weights